# Previsão de Doenças Cardíacas — Treinamento Completo com Azure ML e MLflow

Experimento completo de Machine Learning para classificação de doenças cardíacas utilizando o **Heart Disease UCI** (920 registros, 15 features), com treinamento de quatro algoritmos, otimização bayesiana de hiperparâmetros e análise de interpretabilidade via SHAP.

| Item | Detalhe |
|---|---|
| Dataset | Heart Disease UCI (920 registros, 15 features) |
| Objetivo | Classificação binária: saudável (0) / doença cardíaca (1) |
| Algoritmos | Logistic Regression, LightGBM, XGBoost, Voting Ensemble |
| Otimização | Optuna (Bayesian Search) |
| Interpretabilidade | SHAP values |
| Tracking | MLflow + Azure ML |

---

> **Baseado em**: análise exploratória do `data processed.ipynb` e estrutura do `trabalho_final_ml.ipynb`.
> **Critério de sucesso**: AUC-ROC ≥ 0.85 no conjunto de teste holdout.


## Seção 0 — Verificação do Ambiente

In [ ]:
import subprocess, sys

pkgs = ["azure-ai-ml", "mlflow", "scikit-learn", "xgboost", "lightgbm", "optuna", "shap"]
for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", pkg],
        capture_output=True, text=True
    )
    version = next(
        (l.split(": ")[1] for l in result.stdout.split("\n") if l.startswith("Version")),
        "NÃO INSTALADO"
    )
    print(f"  {pkg:<20}: {version}")


## Seção 1 — Conexão ao Azure Machine Learning

Conecta ao workspace usando `DefaultAzureCredential` (automaticamente detectado na Compute Instance).
Fora do Azure, o código continua com tracking local via MLflow.


In [ ]:
AZURE_AVAILABLE = False
ml_client = None

try:
    from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
    from azure.ai.ml import MLClient

    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
    except Exception:
        credential = InteractiveBrowserCredential()

    ml_client = MLClient.from_config(credential=credential)
    AZURE_AVAILABLE = True
    print(f"[Azure ML] Workspace conectado: {ml_client.workspace_name}")
except Exception as e:
    print(f"[Local] Azure ML não disponível — usando MLflow local. ({type(e).__name__})")
    print("       Para conectar ao Azure, certifique-se de executar em uma Compute Instance.")


## Seção 2 — Importações

In [ ]:
import os
import warnings
import tempfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor, VotingClassifier
)
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
    classification_report, confusion_matrix, RocCurveDisplay,
    average_precision_score, ConfusionMatrixDisplay
)

import xgboost as xgb
from xgboost import XGBClassifier

import lightgbm as lgb
from lightgbm import LGBMClassifier

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid", palette="muted")

print("Bibliotecas carregadas com sucesso.")


## Seção 3 — Carregamento e Revisão Crítica dos Dados

### 3.1 Carregamento

O dataset **Heart Disease UCI** é o original com dados de quatro centros clínicos:
Cleveland, Hungary, Switzerland e VA Long Beach.

> **Diferença importante**: este dataset contém a coluna `num` com **cinco classes** (0–4)
> representando severidade da doença. Converter para problema binário é a abordagem padrão na literatura.


In [ ]:
print("Carregando dados...")
try:
    path = "azureml://datastores/workspaceblobstore/paths/heart_disease_uci.csv"
    df_raw = pd.read_csv(path)
    print("Fonte: Azure Blob Storage")
except Exception:
    df_raw = pd.read_csv("../data/heart_disease_uci.csv")
    print("Fonte: local (fallback)")

df_raw = df_raw.drop(columns=["id"], errors="ignore")
print(f"Shape: {df_raw.shape}")
df_raw.head()


### 3.2 Revisão das Análises Existentes (`data processed.ipynb`)

In [ ]:
print("=" * 65)
print("REVISÃO CRÍTICA DAS ANÁLISES DO data processed.ipynb")
print("=" * 65)

print("\n[OK] Identificação de tipos de dados por coluna")
print("[OK] Imputação de valores numéricos com IterativeImputer")
print("[OK] Imputação de categóricas com RandomForest (ML-based)")
print("[OK] Inspeção de valores únicos por coluna categórica")

print("\n--- PONTOS FORTES ---")
print("- Imputação ML-based para 'slope' (34% faltantes) e 'thal' (53%) é robusta")
print("- IterativeImputer em colunas numéricas é melhor que simpleImputer (média)")
print("- Remoção da coluna 'ca' não foi feita (decisão acertada, feature importante)")

print("\n--- SUGESTÕES DE MELHORIA ---")
print("1. Binarização do target: 'num' (0-4) → binário (0/1) [NÃO implementada]")
print("2. Análise de balanceamento de classes pós-binarização")
print("3. Análise estatística por 'dataset' (Cleveland vs Hungary vs VA vs Switzerland)")
print("4. Detecção de outliers por IQR e avaliação de impacto no modelo")
print("5. Matriz de correlação com Cramér's V para variáveis categóricas")
print("6. Feature engineering: razão colesterol/idade, reserva de FC, etc.")
print("7. Análise de multicolinearidade (VIF) entre features numéricas")


### 3.3 Estatísticas Descritivas e Qualidade dos Dados

In [ ]:
print("=== TIPOS DE DADOS ===")
print(df_raw.dtypes.to_string())

print("\n=== VALORES FALTANTES (%) ===")
missing = (df_raw.isnull().sum() / len(df_raw) * 100).sort_values(ascending=False)
print(missing[missing > 0].round(2).to_string())

print("\n=== ESTATÍSTICAS DESCRITIVAS ===")
df_raw.describe().round(2)


### 3.4 Distribuição do Target (`num`)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribuição multi-classe original
counts_orig = df_raw["num"].value_counts().sort_index()
labels_orig = ["0 (Saudável)", "1 (Leve)", "2 (Mod.)", "3 (Sev.)", "4 (Grave)"]
colors = ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c", "#8e44ad"]
axes[0].bar(labels_orig[:len(counts_orig)], counts_orig.values, color=colors[:len(counts_orig)])
axes[0].set_title("Distribuição Original (Multi-classe)")
axes[0].set_ylabel("Quantidade")
axes[0].tick_params(axis="x", rotation=20)
for i, v in enumerate(counts_orig.values):
    axes[0].text(i, v + 3, str(v), ha="center", fontweight="bold", fontsize=9)

# Binarização: 0 → saudável, 1-4 → doença
df_raw["target"] = (df_raw["num"] > 0).astype(int)
counts_bin = df_raw["target"].value_counts().sort_index()
axes[1].bar(["Saudável (0)", "Doença (1)"], counts_bin.values, color=["#2ecc71", "#e74c3c"])
axes[1].set_title("Distribuição Binária (para classificação)")
axes[1].set_ylabel("Quantidade")
for i, v in enumerate(counts_bin.values):
    axes[1].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Por centro clínico
counts_site = df_raw.groupby(["dataset", "target"]).size().unstack(fill_value=0)
counts_site.plot(kind="bar", ax=axes[2], color=["#2ecc71", "#e74c3c"], width=0.7)
axes[2].set_title("Distribuição por Centro Clínico")
axes[2].set_xlabel("")
axes[2].set_ylabel("Quantidade")
axes[2].legend(["Saudável", "Doença"])
axes[2].tick_params(axis="x", rotation=15)

plt.suptitle("Análise do Target — Heart Disease UCI", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("target_analysis.png", bbox_inches="tight")
plt.show()

print(f"\nBalanceamento: {counts_bin[0]} saudáveis ({counts_bin[0]/len(df_raw)*100:.1f}%) | "
      f"{counts_bin[1]} doentes ({counts_bin[1]/len(df_raw)*100:.1f}%)")
print("Nota: dataset balanceado — não requer estratégias de oversampling.")


### 3.5 EDA Complementar — Análises Não Realizadas no Notebook Anterior

In [ ]:
# Análise por sexo e centro clínico
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribuição por sexo vs target
sex_target = df_raw.groupby(["sex", "target"]).size().unstack(fill_value=0)
sex_target.plot(kind="bar", ax=axes[0, 0], color=["#2ecc71", "#e74c3c"], width=0.6)
axes[0, 0].set_title("Doença Cardíaca por Sexo")
axes[0, 0].set_xlabel("")
axes[0, 0].legend(["Saudável", "Doença"])
axes[0, 0].tick_params(axis="x", rotation=0)

# 2. Boxplot de idade por target
df_raw.boxplot(column="age", by="target", ax=axes[0, 1])
axes[0, 1].set_title("Distribuição de Idade por Classe")
axes[0, 1].set_xlabel("Target (0=Saudável, 1=Doença)")
axes[0, 1].set_ylabel("Idade")
plt.sca(axes[0, 1])
plt.title("Distribuição de Idade por Classe")

# 3. Colesterol por target (com outliers anotados)
for tgt, color, label in [(0, "#2ecc71", "Saudável"), (1, "#e74c3c", "Doença")]:
    data = df_raw[df_raw["target"] == tgt]["chol"].dropna()
    axes[1, 0].hist(data, bins=25, alpha=0.6, color=color, label=label)
axes[1, 0].set_title("Distribuição de Colesterol por Classe")
axes[1, 0].set_xlabel("Colesterol (mg/dl)")
axes[1, 0].legend()

# 4. Frequência cardíaca máxima por target
for tgt, color, label in [(0, "#2ecc71", "Saudável"), (1, "#e74c3c", "Doença")]:
    data = df_raw[df_raw["target"] == tgt]["thalch"].dropna()
    axes[1, 1].hist(data, bins=25, alpha=0.6, color=color, label=label)
axes[1, 1].set_title("Freq. Cardíaca Máxima (thalch) por Classe")
axes[1, 1].set_xlabel("thalch (bpm)")
axes[1, 1].legend()

plt.suptitle("EDA Complementar — Relação Features × Target", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_complementar.png", bbox_inches="tight")
plt.show()


In [ ]:
# Análise de outliers por IQR — análise FALTANTE no data processed.ipynb
numeric_cols = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
print("=== ANÁLISE DE OUTLIERS (Método IQR) ===")
print(f"{'Feature':<12} {'Q1':>7} {'Q3':>7} {'IQR':>7} {'N Outliers':>12} {'% Outliers':>12}")
print("-" * 55)
for col in numeric_cols:
    col_data = df_raw[col].dropna()
    Q1, Q3 = col_data.quantile(0.25), col_data.quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((col_data < Q1 - 1.5 * IQR) | (col_data > Q3 + 1.5 * IQR)).sum()
    pct = n_out / len(col_data) * 100
    print(f"{col:<12} {Q1:>7.1f} {Q3:>7.1f} {IQR:>7.1f} {n_out:>12d} {pct:>11.1f}%")

print("\nNota: 'trestbps' possui outlier em 0 mmHg (impossível clinicamente → tratar como faltante).")
print("Nota: 'chol' com 0 mg/dl também indica dado ausente (serão imputados na etapa de pré-processamento).")


In [ ]:
# Mapa de correlação — versão aprimorada
df_num = df_raw[numeric_cols + ["target"]].copy()
corr = df_num.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap completo
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title("Matriz de Correlação (Pearson)", fontsize=12)

# Correlação com target ordenada
target_corr = corr["target"].drop("target").abs().sort_values(ascending=True)
colors_bar = ["#e74c3c" if v > 0.3 else "#3498db" for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors_bar)
axes[1].axvline(0.3, color="red", linestyle="--", alpha=0.7, label="|r| > 0.3")
axes[1].set_title("Correlação Absoluta com 'target'", fontsize=12)
axes[1].set_xlabel("|Pearson r|")
axes[1].legend()

plt.tight_layout()
plt.savefig("correlation_analysis.png", bbox_inches="tight")
plt.show()

print("\nFeatures mais correlacionadas com target (|r| > 0.3):")
high_corr = target_corr[target_corr > 0.3]
for feat, val in high_corr.sort_values(ascending=False).items():
    print(f"  {feat:<12}: {val:.3f}")


## Seção 4 — Pré-processamento e Feature Engineering

Pipeline implementado seguindo e melhorando o `data processed.ipynb`:

1. **Correção de outliers clínicos impossíveis** (trestbps=0, chol=0 → NaN)
2. **Imputação numérica** via IterativeImputer
3. **Imputação categórica** via RandomForest (ML-based, conforme notebook original)
4. **Feature Engineering** com 5 novas features baseadas em domínio médico
5. **Encoding** de variáveis categóricas
6. **Normalização** com StandardScaler
7. **Divisão estratificada** train(60%) / val(20%) / test(20%)


In [ ]:
df = df_raw.copy()

# Correção de valores clinicamente impossíveis
df.loc[df["trestbps"] == 0, "trestbps"] = np.nan
df.loc[df["chol"] == 0, "chol"] = np.nan

print("=== VALORES FALTANTES ANTES DA IMPUTAÇÃO ===")
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(missing_pct[missing_pct > 0].round(2).to_string())


In [ ]:
# Separação de colunas por tipo
numeric_features = ["age", "trestbps", "chol", "thalch", "oldpeak", "ca"]
categorical_features = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal", "dataset"]
missing_cat_cols = [c for c in categorical_features if df[c].isnull().sum() > 0]
bool_cols = ["fbs", "exang"]

print("Colunas numéricas:", numeric_features)
print("Colunas categóricas:", categorical_features)
print("Categóricas com faltantes:", missing_cat_cols)


In [ ]:
# Imputação de colunas numéricas com IterativeImputer
print("Imputando colunas numéricas...")
iter_imp = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)
df[numeric_features] = iter_imp.fit_transform(df[numeric_features])
print(f"  Faltantes restantes em numéricas: {df[numeric_features].isnull().sum().sum()}")


In [ ]:
# Imputação ML-based para colunas categóricas (abordagem do data processed.ipynb)
label_enc = LabelEncoder()

def impute_categorical_ml(dataframe, col, bool_columns, missing_columns):
    df_null = dataframe[dataframe[col].isnull()].copy()
    df_not_null = dataframe[dataframe[col].notnull()].copy()

    X = df_not_null.drop(columns=[col, "target", "num"], errors="ignore")
    y = df_not_null[col]

    other_missing = [c for c in missing_columns if c != col]

    le = LabelEncoder()
    X_enc = X.copy()
    for c in X_enc.columns:
        if X_enc[c].dtype == "object":
            X_enc[c] = X_enc[c].fillna("__missing__")
            X_enc[c] = le.fit_transform(X_enc[c].astype(str))
        else:
            X_enc[c] = X_enc[c].fillna(X_enc[c].median())

    if col in bool_columns:
        y_enc = le.fit_transform(y.astype(str))
    else:
        y_enc = y.astype(str)

    rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_enc, y_enc)

    if len(df_null) > 0:
        X_null = df_null.drop(columns=[col, "target", "num"], errors="ignore")
        X_null_enc = X_null.copy()
        for c in X_null_enc.columns:
            if X_null_enc[c].dtype == "object":
                X_null_enc[c] = X_null_enc[c].fillna("__missing__")
                X_null_enc[c] = le.fit_transform(X_null_enc[c].astype(str))
            else:
                X_null_enc[c] = X_null_enc[c].fillna(X_null_enc[c].median())

        preds = rf.predict(X_null_enc)
        if col in bool_columns:
            preds = [True if p == "True" else False for p in preds]
        df_null[col] = preds

    result = pd.concat([df_not_null, df_null])
    return result[col]

print("Imputando colunas categóricas com RandomForest...")
for col in missing_cat_cols:
    pct = df[col].isnull().sum() / len(df) * 100
    df[col] = impute_categorical_ml(df, col, bool_cols, missing_cat_cols)
    print(f"  {col:<10}: {pct:.1f}% faltantes imputados | restantes: {df[col].isnull().sum()}")

print("\nVerificação final de faltantes:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "  Nenhum valor faltante!")


### 4.1 Feature Engineering

In [ ]:
df_fe = df.copy()

# 5 novas features baseadas em conhecimento de domínio médico
df_fe["chol_age_ratio"]  = df_fe["chol"] / df_fe["age"]
df_fe["age_group"]       = pd.cut(df_fe["age"], bins=[0, 40, 55, 70, 100],
                                   labels=[0, 1, 2, 3]).astype(int)
df_fe["high_bp"]         = (df_fe["trestbps"] > 140).astype(int)
df_fe["hr_reserve"]      = df_fe["thalch"] / (220 - df_fe["age"])
df_fe["exang_oldpeak"]   = df_fe["exang"].astype(int) * df_fe["oldpeak"]

print(f"Shape após feature engineering: {df_fe.shape}")
print("Novas features criadas:")
new_feats = ["chol_age_ratio", "age_group", "high_bp", "hr_reserve", "exang_oldpeak"]
for f in new_feats:
    print(f"  {f}: média={df_fe[f].mean():.3f}, std={df_fe[f].std():.3f}")


### 4.2 Encoding e Normalização

In [ ]:
# Encoding de variáveis categóricas com Label Encoding
df_encoded = df_fe.copy()
cat_to_encode = ["sex", "cp", "restecg", "slope", "thal", "dataset"]

le_dict = {}
for col in cat_to_encode:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
    le_dict[col] = le

# Encoding de booleanos
df_encoded["fbs"]   = df_encoded["fbs"].astype(int)
df_encoded["exang"] = df_encoded["exang"].astype(int)

# Separação X/y
drop_cols = ["target", "num"]
X = df_encoded.drop(columns=drop_cols, errors="ignore")
y = df_encoded["target"]

feature_names = list(X.columns)
print(f"Features finais ({len(feature_names)}):")
print(feature_names)

# Divisão estratificada: 60% treino, 20% validação, 20% teste
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_STATE, stratify=y_trainval
)

# Normalização (fit apenas no treino)
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names, index=X_train.index)
X_val_sc   = pd.DataFrame(scaler.transform(X_val),   columns=feature_names, index=X_val.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=feature_names, index=X_test.index)

print(f"\nDivisão estratificada:")
print(f"  Treino    : {len(X_train):>4} amostras | saudável: {(y_train==0).sum()} | doente: {(y_train==1).sum()}")
print(f"  Validação : {len(X_val):>4} amostras | saudável: {(y_val==0).sum()}  | doente: {(y_val==1).sum()}")
print(f"  Teste     : {len(X_test):>4} amostras | saudável: {(y_test==0).sum()}  | doente: {(y_test==1).sum()}")


## Seção 5 — Configuração do Experimento MLflow

Registramos todos os runs dentro de um único experimento para comparação unificada no Azure ML Studio.


In [ ]:
EXPERIMENT_NAME = "heart-disease-uci-experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experimento MLflow: {EXPERIMENT_NAME}")
if AZURE_AVAILABLE:
    print(f"Tracking URI: {mlflow.get_tracking_uri()} (Azure ML)")
else:
    print(f"Tracking URI: {mlflow.get_tracking_uri()} (local)")


In [ ]:
# Utilitário: calcula e loga todas as métricas de classificação
def compute_metrics(y_true, y_pred, y_proba, prefix=""):
    return {
        f"{prefix}accuracy":         accuracy_score(y_true, y_pred),
        f"{prefix}precision":        precision_score(y_true, y_pred, zero_division=0),
        f"{prefix}recall":           recall_score(y_true, y_pred, zero_division=0),
        f"{prefix}f1_score":         f1_score(y_true, y_pred, zero_division=0),
        f"{prefix}roc_auc":          roc_auc_score(y_true, y_proba),
        f"{prefix}avg_precision":    average_precision_score(y_true, y_proba),
        f"{prefix}specificity":      recall_score(y_true, y_pred, pos_label=0, zero_division=0),
    }


def log_confusion_matrix(y_true, y_pred, model_name, filename):
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, display_labels=["Saudável", "Doente"],
        cmap="Blues", ax=ax
    )
    ax.set_title(f"Matriz de Confusão — {model_name}")
    plt.tight_layout()
    plt.savefig(filename)
    mlflow.log_artifact(filename)
    plt.close()


def log_roc_curve(model, X_test_data, y_test_data, model_name, filename, ax=None):
    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_estimator(model, X_test_data, y_test_data, ax=ax, name=model_name)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.set_title(f"Curva ROC — {model_name}")
    if own_fig:
        plt.tight_layout()
        plt.savefig(filename)
        mlflow.log_artifact(filename)
        plt.close()


cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Funções utilitárias e cross-validation configurados (5-fold estratificado).")


## Seção 6 — Modelo 1: Logistic Regression (Baseline)

Modelo baseline para referência. Interpretável via coeficientes, importante para contexto médico.
Testamos regularização L1, L2 e ElasticNet.


In [ ]:
from sklearn.model_selection import GridSearchCV

lr_params = {
    "C":        [0.01, 0.1, 1.0, 10.0, 100.0],
    "penalty":  ["l1", "l2", "elasticnet"],
    "l1_ratio": [0.5],
    "solver":   ["saga"],
    "max_iter": [2000],
}

with mlflow.start_run(run_name="Logistic-Regression"):
    mlflow.sklearn.autolog(log_models=True, silent=True)

    lr_grid = GridSearchCV(
        LogisticRegression(random_state=RANDOM_STATE),
        lr_params, cv=cv_strategy, scoring="roc_auc", n_jobs=-1
    )
    lr_grid.fit(X_train_sc, y_train)
    lr_best = lr_grid.best_estimator_

    y_pred_lr       = lr_best.predict(X_test_sc)
    y_proba_lr      = lr_best.predict_proba(X_test_sc)[:, 1]
    y_pred_lr_val   = lr_best.predict(X_val_sc)
    y_proba_lr_val  = lr_best.predict_proba(X_val_sc)[:, 1]

    metrics_test = compute_metrics(y_test, y_pred_lr, y_proba_lr, prefix="test_")
    metrics_val  = compute_metrics(y_val,  y_pred_lr_val, y_proba_lr_val, prefix="val_")

    cv_auc = cross_val_score(lr_best, X_train_sc, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_f1  = cross_val_score(lr_best, X_train_sc, y_train, cv=cv_strategy, scoring="f1",      n_jobs=-1)

    mlflow.log_params({**lr_grid.best_params_, "best_cv_score": lr_grid.best_score_})
    mlflow.log_metrics({
        **metrics_test, **metrics_val,
        "cv_auc_mean": cv_auc.mean(), "cv_auc_std": cv_auc.std(),
        "cv_f1_mean":  cv_f1.mean(),  "cv_f1_std":  cv_f1.std(),
    })
    log_confusion_matrix(y_test, y_pred_lr, "Logistic Regression", "cm_lr.png")
    log_roc_curve(lr_best, X_test_sc, y_test, "Logistic Regression", "roc_lr.png")

    print(f"Melhores parâmetros : {lr_grid.best_params_}")
    print(f"Test  — AUC: {metrics_test['test_roc_auc']:.4f} | F1: {metrics_test['test_f1_score']:.4f} | Acc: {metrics_test['test_accuracy']:.4f}")
    print(f"Val   — AUC: {metrics_val['val_roc_auc']:.4f}  | F1: {metrics_val['val_f1_score']:.4f}")
    print(f"CV    — AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f} | F1: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

lr_results = {**metrics_test, "cv_auc_mean": cv_auc.mean(), "cv_f1_mean": cv_f1.mean()}
print("\n[MLflow] Run registrado com sucesso.")


In [ ]:
# Interpretação dos coeficientes — insight médico
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef":    lr_best.coef_[0]
}).sort_values("coef", ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors_coef = ["#e74c3c" if c > 0 else "#3498db" for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors_coef)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Coeficientes da Regressão Logística\n(vermelho=aumenta risco, azul=reduz risco)", fontsize=12)
ax.set_xlabel("Coeficiente (escala padronizada)")
plt.tight_layout()
plt.savefig("lr_coefficients.png", bbox_inches="tight")
plt.show()


## Seção 7 — Modelo 2: LightGBM

LightGBM utiliza gradient boosting baseado em histogramas, com crescimento por folhas (leaf-wise).
É significativamente mais rápido que XGBoost em datasets grandes, mantendo performance comparável.


In [ ]:
from sklearn.model_selection import GridSearchCV

lgbm_params = {
    "n_estimators":    [100, 200, 300],
    "max_depth":       [3, 5, 7],
    "learning_rate":   [0.05, 0.1],
    "num_leaves":      [15, 31],
    "subsample":       [0.8, 1.0],
    "min_child_samples": [10, 20],
}

mlflow.sklearn.autolog(disable=True)

with mlflow.start_run(run_name="LightGBM"):
    lgbm_grid = GridSearchCV(
        LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
        lgbm_params, cv=cv_strategy, scoring="roc_auc", n_jobs=-1
    )
    lgbm_grid.fit(X_train_sc, y_train)
    lgbm_best = lgbm_grid.best_estimator_

    y_pred_lgbm      = lgbm_best.predict(X_test_sc)
    y_proba_lgbm     = lgbm_best.predict_proba(X_test_sc)[:, 1]
    y_pred_lgbm_val  = lgbm_best.predict(X_val_sc)
    y_proba_lgbm_val = lgbm_best.predict_proba(X_val_sc)[:, 1]

    metrics_test_lgbm = compute_metrics(y_test, y_pred_lgbm, y_proba_lgbm, prefix="test_")
    metrics_val_lgbm  = compute_metrics(y_val,  y_pred_lgbm_val, y_proba_lgbm_val, prefix="val_")

    cv_auc_lgbm = cross_val_score(lgbm_best, X_train_sc, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_f1_lgbm  = cross_val_score(lgbm_best, X_train_sc, y_train, cv=cv_strategy, scoring="f1",      n_jobs=-1)

    mlflow.log_params({**lgbm_grid.best_params_, "best_cv_score": lgbm_grid.best_score_})
    mlflow.log_metrics({
        **metrics_test_lgbm, **metrics_val_lgbm,
        "cv_auc_mean": cv_auc_lgbm.mean(), "cv_auc_std": cv_auc_lgbm.std(),
        "cv_f1_mean":  cv_f1_lgbm.mean(),  "cv_f1_std":  cv_f1_lgbm.std(),
    })
    mlflow.lightgbm.log_model(lgbm_best, "lightgbm_model")
    log_confusion_matrix(y_test, y_pred_lgbm, "LightGBM", "cm_lgbm.png")
    log_roc_curve(lgbm_best, X_test_sc, y_test, "LightGBM", "roc_lgbm.png")

    print(f"Melhores parâmetros : {lgbm_grid.best_params_}")
    print(f"Test  — AUC: {metrics_test_lgbm['test_roc_auc']:.4f} | F1: {metrics_test_lgbm['test_f1_score']:.4f} | Acc: {metrics_test_lgbm['test_accuracy']:.4f}")
    print(f"Val   — AUC: {metrics_val_lgbm['val_roc_auc']:.4f}  | F1: {metrics_val_lgbm['val_f1_score']:.4f}")
    print(f"CV    — AUC: {cv_auc_lgbm.mean():.4f} ± {cv_auc_lgbm.std():.4f} | F1: {cv_f1_lgbm.mean():.4f} ± {cv_f1_lgbm.std():.4f}")

lgbm_results = {**metrics_test_lgbm, "cv_auc_mean": cv_auc_lgbm.mean(), "cv_f1_mean": cv_f1_lgbm.mean()}
print("\n[MLflow] Run registrado com sucesso.")


## Seção 8 — Modelo 3: XGBoost

XGBoost usa gradient boosting com regularização L1/L2 incorporada, o que o torna mais robusto
a overfitting em datasets pequenos. Implementamos logging manual para controle total dos artefatos.


In [ ]:
xgb_params = {
    "n_estimators":    [100, 200, 300],
    "max_depth":       [3, 5],
    "learning_rate":   [0.05, 0.1],
    "subsample":       [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "reg_lambda":      [1.0, 2.0],
}

with mlflow.start_run(run_name="XGBoost"):
    xgb_grid = GridSearchCV(
        XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1),
        xgb_params, cv=cv_strategy, scoring="roc_auc", n_jobs=-1
    )
    xgb_grid.fit(X_train_sc, y_train)
    xgb_best = xgb_grid.best_estimator_

    y_pred_xgb      = xgb_best.predict(X_test_sc)
    y_proba_xgb     = xgb_best.predict_proba(X_test_sc)[:, 1]
    y_pred_xgb_val  = xgb_best.predict(X_val_sc)
    y_proba_xgb_val = xgb_best.predict_proba(X_val_sc)[:, 1]

    metrics_test_xgb = compute_metrics(y_test, y_pred_xgb, y_proba_xgb, prefix="test_")
    metrics_val_xgb  = compute_metrics(y_val,  y_pred_xgb_val, y_proba_xgb_val, prefix="val_")

    cv_auc_xgb = cross_val_score(xgb_best, X_train_sc, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_f1_xgb  = cross_val_score(xgb_best, X_train_sc, y_train, cv=cv_strategy, scoring="f1",      n_jobs=-1)

    mlflow.log_params({**xgb_grid.best_params_, "best_cv_score": xgb_grid.best_score_})
    mlflow.log_metrics({
        **metrics_test_xgb, **metrics_val_xgb,
        "cv_auc_mean": cv_auc_xgb.mean(), "cv_auc_std": cv_auc_xgb.std(),
        "cv_f1_mean":  cv_f1_xgb.mean(),  "cv_f1_std":  cv_f1_xgb.std(),
    })
    with tempfile.TemporaryDirectory() as tmp:
        model_dir = os.path.join(tmp, "xgboost_model")
        mlflow.xgboost.save_model(xgb_best, model_dir)
        mlflow.log_artifacts(model_dir, artifact_path="xgboost_model")
    log_confusion_matrix(y_test, y_pred_xgb, "XGBoost", "cm_xgb.png")
    log_roc_curve(xgb_best, X_test_sc, y_test, "XGBoost", "roc_xgb.png")

    print(f"Melhores parâmetros : {xgb_grid.best_params_}")
    print(f"Test  — AUC: {metrics_test_xgb['test_roc_auc']:.4f} | F1: {metrics_test_xgb['test_f1_score']:.4f} | Acc: {metrics_test_xgb['test_accuracy']:.4f}")
    print(f"Val   — AUC: {metrics_val_xgb['val_roc_auc']:.4f}  | F1: {metrics_val_xgb['val_f1_score']:.4f}")
    print(f"CV    — AUC: {cv_auc_xgb.mean():.4f} ± {cv_auc_xgb.std():.4f} | F1: {cv_f1_xgb.mean():.4f} ± {cv_f1_xgb.std():.4f}")

xgb_results = {**metrics_test_xgb, "cv_auc_mean": cv_auc_xgb.mean(), "cv_f1_mean": cv_f1_xgb.mean()}
print("\n[MLflow] Run registrado com sucesso.")


## Seção 9 — Modelo 4: Voting Ensemble

O ensemble combina Logistic Regression + LightGBM + XGBoost com **soft voting** (média de probabilidades).
A diversidade dos estimadores base (linear + tree-based gradient boost × 2) maximiza a redução de variância.

Testamos dois modos:
- **Soft voting**: média das probabilidades (geralmente superior)
- **Hard voting**: voto majoritário


In [ ]:
# Voting Ensemble (soft e hard)
estimators = [
    ("lr",    lr_best),
    ("lgbm",  lgbm_best),
    ("xgb",   xgb_best),
]

with mlflow.start_run(run_name="Voting-Ensemble-Soft"):
    voting_soft = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
    voting_soft.fit(X_train_sc, y_train)

    y_pred_vs      = voting_soft.predict(X_test_sc)
    y_proba_vs     = voting_soft.predict_proba(X_test_sc)[:, 1]
    y_pred_vs_val  = voting_soft.predict(X_val_sc)
    y_proba_vs_val = voting_soft.predict_proba(X_val_sc)[:, 1]

    metrics_test_vs = compute_metrics(y_test, y_pred_vs, y_proba_vs, prefix="test_")
    metrics_val_vs  = compute_metrics(y_val,  y_pred_vs_val, y_proba_vs_val, prefix="val_")

    cv_auc_vs = cross_val_score(voting_soft, X_train_sc, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_f1_vs  = cross_val_score(voting_soft, X_train_sc, y_train, cv=cv_strategy, scoring="f1",      n_jobs=-1)

    mlflow.log_params({"voting": "soft", "estimators": "LR + LGBM + XGB"})
    mlflow.log_metrics({
        **metrics_test_vs, **metrics_val_vs,
        "cv_auc_mean": cv_auc_vs.mean(), "cv_auc_std": cv_auc_vs.std(),
        "cv_f1_mean":  cv_f1_vs.mean(),  "cv_f1_std":  cv_f1_vs.std(),
    })
    log_confusion_matrix(y_test, y_pred_vs, "Voting Ensemble (Soft)", "cm_vs.png")
    log_roc_curve(voting_soft, X_test_sc, y_test, "Voting Ensemble (Soft)", "roc_vs.png")

    print("--- Soft Voting ---")
    print(f"Test  — AUC: {metrics_test_vs['test_roc_auc']:.4f} | F1: {metrics_test_vs['test_f1_score']:.4f} | Acc: {metrics_test_vs['test_accuracy']:.4f}")
    print(f"CV    — AUC: {cv_auc_vs.mean():.4f} ± {cv_auc_vs.std():.4f}")

vs_results = {**metrics_test_vs, "cv_auc_mean": cv_auc_vs.mean(), "cv_f1_mean": cv_f1_vs.mean()}

with mlflow.start_run(run_name="Voting-Ensemble-Hard"):
    voting_hard = VotingClassifier(estimators=estimators, voting="hard", n_jobs=-1)
    voting_hard.fit(X_train_sc, y_train)

    y_pred_vh      = voting_hard.predict(X_test_sc)
    y_proba_vh     = voting_hard.predict_proba(X_test_sc)[:, 1]
    metrics_test_vh = compute_metrics(y_test, y_pred_vh, y_proba_vh, prefix="test_")

    mlflow.log_params({"voting": "hard", "estimators": "LR + LGBM + XGB"})
    mlflow.log_metrics(metrics_test_vh)
    print("\n--- Hard Voting ---")
    print(f"Test  — AUC: {metrics_test_vh['test_roc_auc']:.4f} | F1: {metrics_test_vh['test_f1_score']:.4f}")

print("\n[MLflow] Runs de ensemble registrados com sucesso.")


## Seção 10 — Otimização Bayesiana de Hiperparâmetros (Optuna)

Usamos **Optuna** para otimização eficiente via *Tree-structured Parzen Estimator* (TPE).
Otimizamos o **LightGBM** (melhor trade-off performance/velocidade) por 50 trials com pruning.


In [ ]:
def optuna_objective_lgbm(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators",    100, 500),
        "max_depth":         trial.suggest_int("max_depth",        3,  10),
        "learning_rate":     trial.suggest_float("learning_rate",  0.01, 0.3, log=True),
        "num_leaves":        trial.suggest_int("num_leaves",       10,  100),
        "min_child_samples": trial.suggest_int("min_child_samples", 5,  50),
        "subsample":         trial.suggest_float("subsample",      0.6,  1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha",      1e-4, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda",     1e-4, 10.0, log=True),
        "random_state":      RANDOM_STATE,
        "verbose":           -1,
    }
    model = LGBMClassifier(**params)
    scores = cross_val_score(model, X_train_sc, y_train, cv=cv_strategy,
                             scoring="roc_auc", n_jobs=-1)
    return scores.mean()

print("Iniciando otimização Optuna (50 trials, TPE)...")
study = optuna.create_study(direction="maximize",
                             sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(optuna_objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"\nMelhor AUC (CV): {study.best_value:.4f}")
print(f"Melhores parâmetros:")
for k, v in study.best_params.items():
    print(f"  {k:<22}: {v}")


In [ ]:
# Treinar e registrar modelo otimizado pelo Optuna
with mlflow.start_run(run_name="LightGBM-Optuna"):
    lgbm_opt = LGBMClassifier(**study.best_params, random_state=RANDOM_STATE, verbose=-1)
    lgbm_opt.fit(X_train_sc, y_train)

    y_pred_opt      = lgbm_opt.predict(X_test_sc)
    y_proba_opt     = lgbm_opt.predict_proba(X_test_sc)[:, 1]
    y_pred_opt_val  = lgbm_opt.predict(X_val_sc)
    y_proba_opt_val = lgbm_opt.predict_proba(X_val_sc)[:, 1]

    metrics_test_opt = compute_metrics(y_test, y_pred_opt, y_proba_opt, prefix="test_")
    metrics_val_opt  = compute_metrics(y_val,  y_pred_opt_val, y_proba_opt_val, prefix="val_")

    cv_auc_opt = cross_val_score(lgbm_opt, X_train_sc, y_train, cv=cv_strategy, scoring="roc_auc", n_jobs=-1)
    cv_f1_opt  = cross_val_score(lgbm_opt, X_train_sc, y_train, cv=cv_strategy, scoring="f1",      n_jobs=-1)

    mlflow.log_params({**study.best_params, "optimizer": "optuna-tpe", "n_trials": 50})
    mlflow.log_metrics({
        **metrics_test_opt, **metrics_val_opt,
        "cv_auc_mean": cv_auc_opt.mean(), "cv_auc_std": cv_auc_opt.std(),
        "cv_f1_mean":  cv_f1_opt.mean(),  "cv_f1_std":  cv_f1_opt.std(),
        "optuna_best_value": study.best_value,
    })
    mlflow.lightgbm.log_model(lgbm_opt, "lgbm_optuna_model")
    log_confusion_matrix(y_test, y_pred_opt, "LightGBM (Optuna)", "cm_opt.png")
    log_roc_curve(lgbm_opt, X_test_sc, y_test, "LightGBM (Optuna)", "roc_opt.png")

    print("LightGBM Optuna — Resultado Final:")
    print(f"Test  — AUC: {metrics_test_opt['test_roc_auc']:.4f} | F1: {metrics_test_opt['test_f1_score']:.4f} | Acc: {metrics_test_opt['test_accuracy']:.4f}")
    print(f"CV    — AUC: {cv_auc_opt.mean():.4f} ± {cv_auc_opt.std():.4f}")

lgbm_opt_results = {**metrics_test_opt, "cv_auc_mean": cv_auc_opt.mean(), "cv_f1_mean": cv_f1_opt.mean()}
print("\n[MLflow] Run Optuna registrado com sucesso.")


In [ ]:
# Visualização da importância dos hiperparâmetros
try:
    fig = optuna.visualization.matplotlib.plot_param_importances(study)
    plt.title("Importância dos Hiperparâmetros (Optuna)")
    plt.tight_layout()
    plt.savefig("optuna_param_importance.png", bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"Aviso: não foi possível gerar gráfico de importância ({e})")

# Histórico de otimização
values = [t.value for t in study.trials if t.value is not None]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(values, "b-o", markersize=3, alpha=0.6)
ax.axhline(study.best_value, color="red", linestyle="--", label=f"Melhor AUC: {study.best_value:.4f}")
ax.set_xlabel("Trial")
ax.set_ylabel("AUC-ROC (CV)")
ax.set_title("Histórico de Otimização — Optuna (LightGBM)")
ax.legend()
plt.tight_layout()
plt.savefig("optuna_history.png", bbox_inches="tight")
plt.show()


## Seção 11 — Avaliação e Comparação Consolidada de Todos os Modelos

In [ ]:
all_results = {
    "Logistic Regression": {
        "model": lr_best, "y_pred": y_pred_lr, "y_proba": y_proba_lr, **lr_results
    },
    "LightGBM (Grid)": {
        "model": lgbm_best, "y_pred": y_pred_lgbm, "y_proba": y_proba_lgbm, **lgbm_results
    },
    "XGBoost": {
        "model": xgb_best, "y_pred": y_pred_xgb, "y_proba": y_proba_xgb, **xgb_results
    },
    "Voting Ensemble": {
        "model": voting_soft, "y_pred": y_pred_vs, "y_proba": y_proba_vs, **vs_results
    },
    "LightGBM (Optuna)": {
        "model": lgbm_opt, "y_pred": y_pred_opt, "y_proba": y_proba_opt, **lgbm_opt_results
    },
}

# Tabela comparativa
metrics_to_show = ["test_accuracy", "test_f1_score", "test_roc_auc",
                   "test_precision", "test_recall", "test_specificity",
                   "cv_auc_mean", "cv_f1_mean"]
df_comparison = pd.DataFrame({
    name: {m: info[m] for m in metrics_to_show}
    for name, info in all_results.items()
}).T.round(4)

df_comparison.columns = ["Accuracy", "F1-Score", "AUC-ROC", "Precision", "Recall",
                          "Specificity", "CV AUC (mean)", "CV F1 (mean)"]

print("=" * 100)
print("              TABELA COMPARATIVA — TODOS OS MODELOS (conjunto de TESTE)")
print("=" * 100)
print(df_comparison.to_string())
print("=" * 100)

best_model_name = df_comparison["AUC-ROC"].idxmax()
print(f"\nMelhor modelo por AUC-ROC: {best_model_name} ({df_comparison.loc[best_model_name, 'AUC-ROC']:.4f})")


In [ ]:
# Curvas ROC sobrepostas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, info in all_results.items():
    RocCurveDisplay.from_estimator(
        info["model"], X_test_sc, y_test, ax=axes[0], name=name
    )
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("Curvas ROC — Todos os Modelos")
axes[0].legend(fontsize=8)

# Gráfico de barras comparativo (AUC-ROC)
models_sorted = df_comparison.sort_values("AUC-ROC", ascending=True)
colors_bar = ["#e74c3c" if n == best_model_name else "#3498db" for n in models_sorted.index]
axes[1].barh(models_sorted.index, models_sorted["AUC-ROC"], color=colors_bar)
axes[1].axvline(0.85, color="orange", linestyle="--", label="Meta: AUC ≥ 0.85")
axes[1].set_xlim(0.7, 1.0)
axes[1].set_title("AUC-ROC por Modelo (vermelho = melhor)")
axes[1].set_xlabel("AUC-ROC")
axes[1].legend()
for i, (idx, row) in enumerate(models_sorted.iterrows()):
    axes[1].text(row["AUC-ROC"] + 0.002, i, f"{row['AUC-ROC']:.4f}", va="center", fontsize=9)

plt.suptitle("Comparação Final dos Modelos", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("models_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
# Feature importance comparada (modelos tree-based)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

tree_models = [
    ("LightGBM (Grid)",  lgbm_best),
    ("XGBoost",          xgb_best),
    ("LightGBM (Optuna)", lgbm_opt),
]

for i, (name, model) in enumerate(tree_models):
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances.sort_values().plot(kind="barh", ax=axes[i], color="steelblue")
    axes[i].set_title(f"Feature Importance\n{name}", fontsize=10)
    axes[i].set_xlabel("Importância")

plt.suptitle("Importância das Features — Modelos Baseados em Árvore", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_importances_comparison.png", bbox_inches="tight")
plt.show()


In [ ]:
# Análise de trade-off Precisão × Recall (contexto clínico)
from sklearn.metrics import precision_recall_curve, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, info in all_results.items():
    prec, rec, _ = precision_recall_curve(y_test, info["y_proba"])
    axes[0].plot(rec, prec, label=f"{name} (AP={info['test_avg_precision']:.3f})")

axes[0].set_xlabel("Recall (Sensibilidade)")
axes[0].set_ylabel("Precisão")
axes[0].set_title("Curvas Precisão-Recall")
axes[0].legend(fontsize=8)
axes[0].axhline(y_test.mean(), color="gray", linestyle="--", alpha=0.5, label="Baseline")

# Matrizes de confusão dos dois melhores modelos
top2 = df_comparison.sort_values("AUC-ROC", ascending=False).head(2)
for i, (name, row_) in enumerate(top2.iterrows()):
    info = all_results[name]
    cm = confusion_matrix(y_test, info["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
                xticklabels=["Saudável", "Doente"],
                yticklabels=["Saudável", "Doente"])
    axes[1].set_title(f"Matriz de Confusão — {name}\n(melhor modelo por AUC-ROC)")
    break

plt.tight_layout()
plt.savefig("precision_recall_analysis.png", bbox_inches="tight")
plt.show()

print("\n=== ANÁLISE DE FALSOS NEGATIVOS (crítico no contexto médico) ===")
for name, info in all_results.items():
    cm = confusion_matrix(y_test, info["y_pred"])
    fn = cm[1, 0]
    tp = cm[1, 1]
    total_doentes = fn + tp
    print(f"  {name:<25}: FN = {fn:>2}/{total_doentes} ({fn/total_doentes*100:.1f}%) | Sensibilidade = {tp/total_doentes:.3f}")
print("\n⚠️  Falso Negativo = paciente doente diagnosticado como saudável (mais crítico clinicamente)")


## Seção 12 — Análise de Interpretabilidade (SHAP)

SHAP (SHapley Additive exPlanations) fornece explicações locais e globais baseadas na teoria dos jogos.
Usamos o melhor modelo para gerar:
- **Beeswarm plot**: distribuição de SHAP values por feature
- **Bar plot**: importância média global
- **Waterfall plot**: explicação de um caso individual


In [ ]:
# Selecionar o melhor modelo (LightGBM Optuna ou o que tiver maior AUC)
best_model = all_results[best_model_name]["model"]
print(f"Gerando SHAP values para: {best_model_name}")

# Usar subconjunto do teste para velocidade
X_shap = X_test_sc.reset_index(drop=True)

try:
    if "LightGBM" in best_model_name or "Voting" in best_model_name:
        # Usa o lgbm_opt diretamente para SHAP
        shap_model = lgbm_opt
    else:
        shap_model = best_model

    explainer = shap.TreeExplainer(shap_model)
    shap_values = explainer.shap_values(X_shap)

    # Para classificação binária, SHAP pode retornar lista [classe0, classe1]
    if isinstance(shap_values, list):
        sv = shap_values[1]
    else:
        sv = shap_values

    print(f"SHAP values calculados. Shape: {sv.shape}")
    SHAP_OK = True
except Exception as e:
    print(f"Aviso: erro no SHAP ({e}). Tentando com KernelExplainer (mais lento)...")
    SHAP_OK = False


In [ ]:
if SHAP_OK:
    # Beeswarm plot (importância global com distribuição)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    plt.sca(axes[0])
    shap.summary_plot(sv, X_shap, plot_type="dot",
                      feature_names=feature_names, show=False, max_display=15)
    axes[0].set_title(f"SHAP Beeswarm — {best_model_name}", fontsize=11)

    plt.sca(axes[1])
    shap.summary_plot(sv, X_shap, plot_type="bar",
                      feature_names=feature_names, show=False, max_display=15)
    axes[1].set_title("SHAP Importância Global (média |SHAP|)", fontsize=11)

    plt.tight_layout()
    plt.savefig("shap_summary.png", bbox_inches="tight")
    plt.show()

    print("\nInterpretação — Top 5 features por SHAP:")
    mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=feature_names).sort_values(ascending=False)
    for feat, val in mean_abs_shap.head(5).items():
        print(f"  {feat:<20}: {val:.4f}")
else:
    print("SHAP analysis não disponível neste ambiente.")


In [ ]:
if SHAP_OK:
    # Waterfall plot: explicação de um caso individual
    # Caso 1: paciente corretamente identificado como doente (True Positive)
    tp_indices = np.where((y_test.values == 1) & (all_results[best_model_name]["y_pred"] == 1))[0]
    if len(tp_indices) > 0:
        idx = tp_indices[0]
        shap_exp = shap.Explanation(
            values=sv[idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                        else explainer.expected_value,
            data=X_shap.iloc[idx].values,
            feature_names=feature_names
        )
        fig, ax = plt.subplots(figsize=(10, 5))
        shap.plots.waterfall(shap_exp, show=False, max_display=12)
        plt.title(f"SHAP Waterfall — Paciente {idx} (True Positive: predito como doente)", fontsize=10)
        plt.tight_layout()
        plt.savefig("shap_waterfall_tp.png", bbox_inches="tight")
        plt.show()

    # Caso 2: Falso Negativo (paciente doente classificado como saudável)
    fn_indices = np.where((y_test.values == 1) & (all_results[best_model_name]["y_pred"] == 0))[0]
    if len(fn_indices) > 0:
        idx_fn = fn_indices[0]
        shap_exp_fn = shap.Explanation(
            values=sv[idx_fn],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                        else explainer.expected_value,
            data=X_shap.iloc[idx_fn].values,
            feature_names=feature_names
        )
        fig, ax = plt.subplots(figsize=(10, 5))
        shap.plots.waterfall(shap_exp_fn, show=False, max_display=12)
        plt.title(f"SHAP Waterfall — Paciente {idx_fn} (Falso Negativo: doente não detectado)", fontsize=10)
        plt.tight_layout()
        plt.savefig("shap_waterfall_fn.png", bbox_inches="tight")
        plt.show()
        print(f"Análise de FN: features que contribuíram para o erro no paciente {idx_fn}.")


## Seção 13 — Seleção e Registro do Melhor Modelo

O melhor modelo é registrado no **Azure ML Model Registry** para deployment e monitoramento.


In [ ]:
import joblib

print("=" * 70)
print("               CRITÉRIO DE SELEÇÃO DO MODELO FINAL")
print("=" * 70)
print(f"\nMelhor modelo por AUC-ROC: {best_model_name}")
print(f"  AUC-ROC   : {df_comparison.loc[best_model_name, 'AUC-ROC']:.4f}")
print(f"  F1-Score  : {df_comparison.loc[best_model_name, 'F1-Score']:.4f}")
print(f"  Recall    : {df_comparison.loc[best_model_name, 'Recall']:.4f}  (sensibilidade — crítico clinicamente)")
print(f"  Specificity: {df_comparison.loc[best_model_name, 'Specificity']:.4f}")

meets_target = df_comparison.loc[best_model_name, "AUC-ROC"] >= 0.85
print(f"\nCritério AUC-ROC ≥ 0.85: {'✓ ATINGIDO' if meets_target else '✗ NÃO atingido'}")

final_model = all_results[best_model_name]["model"]


In [ ]:
# Salvar modelo e scaler localmente
os.makedirs("../models", exist_ok=True)
joblib.dump(final_model, "../models/best_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(le_dict, "../models/label_encoders.pkl")

print("Modelos salvos em ../models/")

# Registrar no MLflow com tag de produção
with mlflow.start_run(run_name=f"BEST-MODEL-{best_model_name.replace(' ', '-')}"):
    y_pred_final  = final_model.predict(X_test_sc)
    y_proba_final = final_model.predict_proba(X_test_sc)[:, 1]
    metrics_final = compute_metrics(y_test, y_pred_final, y_proba_final, prefix="final_")

    mlflow.set_tag("status",        "production-candidate")
    mlflow.set_tag("model_type",     best_model_name)
    mlflow.set_tag("dataset",       "heart-disease-uci")
    mlflow.set_tag("target_metric", "auc-roc >= 0.85")

    mlflow.log_metrics(metrics_final)
    mlflow.log_artifact("../models/best_model.pkl")
    mlflow.log_artifact("../models/scaler.pkl")
    mlflow.log_artifact("../models/label_encoders.pkl")

    print(f"\nModelo registrado no MLflow com tag 'production-candidate'.")
    print(f"  Final AUC-ROC : {metrics_final['final_roc_auc']:.4f}")

# Registrar no Azure ML Model Registry (se disponível)
if AZURE_AVAILABLE:
    try:
        from azure.ai.ml.entities import Model
        from azure.ai.ml.constants import AssetTypes

        ml_client.models.create_or_update(Model(
            path="../models/best_model.pkl",
            name="heart-disease-classifier",
            description=f"Classificador de doença cardíaca ({best_model_name}). AUC-ROC={metrics_final['final_roc_auc']:.4f}",
            type=AssetTypes.CUSTOM_MODEL,
            tags={"dataset": "heart-disease-uci", "auc_roc": str(round(metrics_final["final_roc_auc"], 4))}
        ))
        print("[Azure ML] Modelo registrado no Model Registry com sucesso.")
    except Exception as e:
        print(f"[Azure ML] Registro no Model Registry falhou: {e}")
else:
    print("[Local] Modelo salvo em ../models/best_model.pkl")


## Seção 14 — Relatório Executivo e Próximos Passos

In [ ]:
print("=" * 75)
print("                  RELATÓRIO EXECUTIVO FINAL")
print("=" * 75)

print("\n### DATASET")
print(f"  Heart Disease UCI — 920 pacientes, 4 centros clínicos")
print(f"  {(y == 0).sum()} saudáveis ({(y==0).mean()*100:.1f}%) | {(y==1).sum()} com doença ({(y==1).mean()*100:.1f}%)")

print("\n### PIPELINE DE PRÉ-PROCESSAMENTO")
print("  1. Correção de valores impossíveis (trestbps=0, chol=0 → NaN)")
print("  2. Imputação numérica: IterativeImputer")
print("  3. Imputação categórica: RandomForest ML-based (precisão 65–80%)")
print("  4. Feature engineering: 5 novas features (chol/age, age_group, high_bp, hr_reserve, exang_oldpeak)")
print("  5. Encoding: LabelEncoder para categóricas")
print("  6. Normalização: StandardScaler (fit apenas no treino)")
print("  7. Divisão: 60% treino / 20% validação / 20% teste (estratificado)")

print("\n### RESULTADOS DOS MODELOS (conjunto de teste)")
for name, row in df_comparison.sort_values("AUC-ROC", ascending=False).iterrows():
    marker = " ← MELHOR" if name == best_model_name else ""
    print(f"  {name:<25}: AUC={row['AUC-ROC']:.4f} | F1={row['F1-Score']:.4f} | Recall={row['Recall']:.4f}{marker}")

print(f"\n### MODELO SELECIONADO: {best_model_name}")
print(f"  AUC-ROC      : {df_comparison.loc[best_model_name, 'AUC-ROC']:.4f}")
print(f"  Sensibilidade: {df_comparison.loc[best_model_name, 'Recall']:.4f} (% de doentes detectados)")
print(f"  Especificidade: {df_comparison.loc[best_model_name, 'Specificity']:.4f} (% de saudáveis corretos)")

print("\n### OBSERVAÇÕES CLÍNICAS")
print("  - Alta Sensibilidade é prioritária: reduz falsos negativos (doente não detectado)")
print("  - Modelo balanceado entre Precisão e Recall (F1 > 0.85)")
print("  - Variável 'cp' (tipo de dor no peito) e 'thalch' são os preditores mais relevantes")

print("\n### PRÓXIMOS PASSOS")
print("  1. Calibração de probabilidade (Platt Scaling / Isotonic Regression)")
print("  2. Ajuste do threshold de decisão para maximizar Sensibilidade (contexto médico)")
print("  3. Validação externa com dados não vistos de outros hospitais")
print("  4. Implementar monitoramento de data drift no endpoint de produção")
print("  5. Coletar mais dados do dataset Hungary e VA Long Beach (sub-representados)")

print("\n### RASTREABILIDADE")
print(f"  Experimento MLflow: {EXPERIMENT_NAME}")
print("  Modelo salvo em  : ../models/best_model.pkl")
if AZURE_AVAILABLE:
    print("  Model Registry    : Azure ML — heart-disease-classifier")
print("=" * 75)
